In [0]:
%run /Repos/molugurikoushik68@gmail.com/banking-agent/banking-agent/notebooks/09_banking_agent_orchestrator


In [0]:
from typing import Dict

"""
BANKING AGENT - INTELLIGENT ROUTING CHATBOT v3.1

Smart intent detection and routing:
- BALANCE QUERY: Show actual balance (no LLM needed)
- POLICY QUERY: Use hybrid search + LLM
- TRANSACTION QUERY: Validate transaction + show result

Copy-paste this ENTIRE code into ONE cell and run
"""

# ============================================
# WIDGET 1: User ID
# ============================================

dbutils.widgets.text("User ID", "user_050")
user_id_input = dbutils.widgets.get("User ID")

# ============================================
# WIDGET 2: Question
# ============================================

dbutils.widgets.text("Ask a question:", "")
user_query = dbutils.widgets.get("Ask a question:")

# ============================================
# INTENT DETECTION
# ============================================

def detect_intent(user_input: str) -> str:
    """Detect user intent from question"""

    user_lower = user_input.lower()

    # BALANCE INTENT keywords
    balance_keywords = [
        'balance', 'how much', 'account balance', 'my balance',
        'how much money', 'total balance', 'current balance',
        'what is my balance', 'check balance', 'check my balance',
        'how much do i have', 'account funds'
    ]

    # TRANSACTION INTENT keywords
    transaction_keywords = [
        'transfer', 'send', 'pay', 'transaction', 'spend',
        'withdraw', 'deposit', 'move money', 'amount',
        'to amazon', 'to flipkart', 'to google', 'to zomato',
        'to swiggy', 'to paytm', 'upi', 'crypto', 'exchange'
    ]

    # POLICY INTENT keywords
    policy_keywords = [
        'policy', 'limit', 'fee', 'rule', 'how much can',
        'maximum', 'minimum', 'charge', 'cost', 'requirement',
        'what is', 'explain', 'tell me about', 'daily',
        'weekly', 'transfer limit', 'maintenance', 'verification'
    ]

    # Check BALANCE first (most specific)
    for keyword in balance_keywords:
        if keyword in user_lower:
            return "BALANCE"

    # Check TRANSACTION
    for keyword in transaction_keywords:
        if keyword in user_lower:
            return "TRANSACTION"

    # Check POLICY
    for keyword in policy_keywords:
        if keyword in user_lower:
            return "POLICY"

    # Default to POLICY if no match
    return "POLICY"

# ============================================
# INTENT HANDLERS
# ============================================

def handle_balance_query(agent, user_id: str, user_name: str, user_data: Dict) -> Dict:
    """Handle balance inquiry - show actual balance"""

    balance = user_data.get('balance', 0)
    daily_limit = 25000
    daily_used = user_data.get('daily_transfer_used', 0)
    daily_remaining = daily_limit - daily_used

    response = f"""Your current account balance is **₹{balance:,.2f}**.

Daily transfer limit: ₹{daily_limit:,}
Already transferred today: ₹{daily_used:,.2f}
Remaining today: ₹{daily_remaining:,.2f}

Is there anything else you'd like to know about your account?"""

    return {
        'status': 'success',
        'intent': 'BALANCE',
        'response': response,
        'user': user_name,
        'balance': balance,
        'fraud_risk': 'No Transaction',
        'latency_ms': 100
    }

def handle_transaction_query(agent, user_id: str, user_name: str, user_input: str) -> Dict:
    """Handle transaction - validate and show result"""

    print(f"\n[TRANSACTION] Processing: {user_input}")

    # Use the agent's full process_request to validate transaction
    result = agent.process_request(user_id, user_input, f"sess_{user_id}")

    # Add intent type
    result['intent'] = 'TRANSACTION'

    return result

def handle_policy_query(agent, user_id: str, user_name: str, user_input: str) -> Dict:
    """Handle policy inquiry - use hybrid search + LLM"""

    print(f"\n[POLICY] Processing: {user_input}")

    # Use the agent's full process_request
    result = agent.process_request(user_id, user_input, f"sess_{user_id}")

    # Add intent type
    result['intent'] = 'POLICY'

    return result

# ============================================
# MAIN CHATBOT FLOW
# ============================================

if user_id_input and user_query:

    try:
        print(f"\n{'='*70}")
        print(f"INTELLIGENT ROUTING CHATBOT")
        print(f"{'='*70}")
        print(f"\n[INPUT] User: {user_id_input}, Query: {user_query}\n")

        # Step 1: Detect intent
        intent = detect_intent(user_query)
        print(f"[INTENT DETECTED] {intent}\n")

        # Step 2: Route based on intent
        if intent == "BALANCE":
            print("[ROUTE] → Balance Handler")

            # Load user data
            user_data = spark.sql(f"""
            SELECT name, balance
            FROM banking_agent_db.users
            WHERE user_id = '{user_id_input}'
            LIMIT 1
            """).collect()[0]

            # Calculate daily transfer used from transactions
            daily_transfer_result = spark.sql(f"""
            SELECT COALESCE(SUM(amount), 0) as total_used
            FROM banking_agent_db.realistic_transactions
            WHERE user_id = '{user_id_input}'
              AND DATE(timestamp) = CURRENT_DATE()
              AND is_fraudulent = 0
            """).collect()

            daily_used = float(daily_transfer_result[0]['total_used']) if daily_transfer_result else 0.0

            user_name = user_data['name']
            result = handle_balance_query(agent, user_id_input, user_name, {
                'balance': user_data['balance'],
                'daily_transfer_used': daily_used
            })

        elif intent == "TRANSACTION":
            print("[ROUTE] → Transaction Handler")

            # Get user name
            user_data = spark.sql(f"""
            SELECT name FROM banking_agent_db.users
            WHERE user_id = '{user_id_input}' LIMIT 1
            """).collect()[0]

            user_name = user_data['name']
            result = handle_transaction_query(agent, user_id_input, user_name, user_query)

        else:  # POLICY
            print("[ROUTE] → Policy Handler")

            # Get user name
            user_data = spark.sql(f"""
            SELECT name FROM banking_agent_db.users
            WHERE user_id = '{user_id_input}' LIMIT 1
            """).collect()[0]

            user_name = user_data['name']
            result = handle_policy_query(agent, user_id_input, user_name, user_query)

        # Step 3: Display result
        if result['status'] == 'success':

            user_name = result.get('user', user_data['name'] if user_data else 'User')
            user_balance = result.get('balance', 'N/A')
            response = result.get('response', 'No response')
            fraud_risk = result.get('fraud_risk', 'N/A')
            latency = result.get('latency_ms', 'N/A')
            intent_type = result.get('intent', 'UNKNOWN')

            # ============================================
            # BEAUTIFUL DISPLAY
            # ============================================

            displayHTML(f"""
            <div style="font-family: 'Segoe UI', Arial; max-width: 900px; background: white; padding: 25px; border-radius: 12px; border: 1px solid #e0e0e0; box-shadow: 0 2px 8px rgba(0,0,0,0.1);">

                <!-- HEADER -->
                <div style="border-bottom: 2px solid #1976d2; padding-bottom: 15px; margin-bottom: 20px;">
                    <div style="display: flex; align-items: center; justify-content: space-between;">
                        <div>
                            <h2 style="margin: 0; color: #1976d2; font-size: 1.8em;">🏦 Banking Agent</h2>
                            <p style="margin: 8px 0 0 0; color: #666; font-size: 0.95em;">
                                User: <strong>{user_name}</strong> | Balance: <strong>₹{user_balance if isinstance(user_balance, str) else f'{user_balance:,.2f}'}</strong>
                            </p>
                        </div>
                        <div style="background: #e3f2fd; padding: 8px 12px; border-radius: 20px; color: #1976d2; font-weight: bold; font-size: 0.85em;">
                            {intent_type}
                        </div>
                    </div>
                </div>

                <!-- QUESTION -->
                <div style="background: #f5f5f5; padding: 15px; border-radius: 8px; margin-bottom: 15px; border-left: 4px solid #1976d2;">
                    <h4 style="margin: 0 0 8px 0; color: #1976d2; font-size: 0.95em;">📝 Your Question</h4>
                    <p style="margin: 0; font-size: 1.05em; color: #333;">{user_query}</p>
                </div>

                <!-- RESPONSE -->
                <div style="background: #f3e5f5; padding: 15px; border-radius: 8px; margin-bottom: 15px; border-left: 4px solid #7b1fa2;">
                    <h4 style="margin: 0 0 8px 0; color: #7b1fa2; font-size: 0.95em;">💬 Agent Response</h4>
                    <p style="margin: 0; font-size: 1.05em; line-height: 1.7; color: #333;">
                        <strong>Hi {user_name},</strong><br><br>
                        {response}
                    </p>
                </div>

                <!-- METRICS -->
                <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 15px;">

                    <div style="background: {'#f3e5f5' if fraud_risk == 'No Transaction' else '#fff3e0'}; padding: 15px; border-radius: 8px; border: 1px solid {'#ce93d8' if fraud_risk == 'No Transaction' else '#ffb74d'}; text-align: center;">
                        <h4 style="margin: 0 0 8px 0; color: {'#6a1b9a' if fraud_risk == 'No Transaction' else '#e65100'}; font-size: 0.9em;">{'ℹ️ Transaction Status' if fraud_risk == 'No Transaction' else '⚠️ Fraud Risk'}</h4>
                        <p style="margin: 0; font-size: 1.4em; font-weight: bold; color: {'#6a1b9a' if fraud_risk == 'No Transaction' else '#e65100'};">{fraud_risk}</p>
                    </div>

                    <div style="background: #e8f5e9; padding: 15px; border-radius: 8px; border: 1px solid #81c784; text-align: center;">
                        <h4 style="margin: 0 0 8px 0; color: #2e7d32; font-size: 0.9em;">⏱️ Latency</h4>
                        <p style="margin: 0; font-size: 1.4em; font-weight: bold; color: #2e7d32;">{latency if isinstance(latency, str) else f'{latency:.0f}'}ms</p>
                    </div>

                </div>

                <!-- FOOTER -->
                <div style="margin-top: 20px; padding-top: 15px; border-top: 1px solid #e0e0e0; text-align: center; color: #999; font-size: 0.85em;">
                    ✅ Banking Agent v3.1 | Intelligent Routing | Production Ready
                </div>

            </div>
            """)

        else:
            # BLOCKED / ERROR
            reason = result.get('reason', result.get('error', 'Unknown error'))

            displayHTML(f"""
            <div style="font-family: Arial; max-width: 900px; background: #ffebee; padding: 20px; border-radius: 12px; border-left: 4px solid #c62828;">
                <h3 style="margin: 0 0 10px 0; color: #c62828;">❌ Transaction Blocked</h3>
                <p style="margin: 0; font-size: 1.05em; color: #b71c1c;">
                    <strong>Reason:</strong> {reason}
                </p>
            </div>
            """)

    except Exception as e:
        print(f"\n❌ Error: {str(e)}")
        import traceback
        traceback.print_exc()

        displayHTML(f"""
        <div style="font-family: Arial; max-width: 900px; background: #ffebee; padding: 20px; border-radius: 12px; border-left: 4px solid #c62828;">
            <h3 style="margin: 0 0 10px 0; color: #c62828;">❌ Error Processing Request</h3>
            <p style="margin: 0; color: #c62828;">{str(e)}</p>
        </div>
        """)

else:
    # Waiting for input
    if user_id_input and not user_query:
        print("⏳ User ID received. Please enter your question above...")
    elif user_query and not user_id_input:
        print("⏳ Question received. Please enter your User ID above...")
    else:
        print("⏳ Ready to chat! Please enter User ID and your question above...")
